In [13]:
import logging
import os
import sys
import re
import subprocess
from typing import Union, List, Optional

import msgspec
from pathlib import Path
from msgspec.json import format
from pprint import pformat, pprint

encoder = msgspec.json.Encoder()
decoder = msgspec.json.Decoder()

import torch
import torch.nn.functional as F
import numpy as np
import random


temporal_query_type_class_id = {
    "Explicit": 0,
    "Implicit": 1,
    "TemporalAnswer": 2,
}

allen_relation_class_id = {
    "Before": 0,
    "After": 1,
    "Meets": 2,
    "MetBy": 3,
    "Overlaps": 4,
    "OverlappedBy": 5,
    "Starts": 6,
    "StartedBy": 7,
    "During": 8,
    "Contains": 9,
    "Finishes": 10,
    "FinishedBy": 11,
    "Equals": 12,
    "Empty": 13,
}



def norm(x):
    return F.normalize(x, p=2, dim=-1)


# Compile once at module load time for speed
_WHITESPACE_RE = re.compile(r"\s+")


def normalize_cmd(cmd: str) -> str:
    """Normalize whitespace and remove newlines in a shell command."""
    cmd = _WHITESPACE_RE.sub(" ", cmd.strip())
    cmd = cmd.replace("\n", " ")
    return cmd


def run(
    cmd: Union[str, List[str]], env: Optional[dict] = None, dry_run: bool = False
) -> None:
    """
    Run one or multiple shell commands safely.

    Args:
        cmd: A single command string or a list of commands.
        env: Optional environment variables to pass to subprocess.
        dry_run: If True, prints commands instead of executing them.
    """
    if isinstance(cmd, list):
        cmd_list = [normalize_cmd(c) for c in cmd if c.strip()]
        joined_cmd = " && ".join(cmd_list)
    else:
        joined_cmd = normalize_cmd(cmd)

    # print(f"\n>>> Running:\n{joined_cmd}\n", flush=True)

    if dry_run:
        return

    try:
        # pprint(dict(os.environ))
        pprint(joined_cmd)
        subprocess.run(
            joined_cmd, 
            # timeout=15,
            shell=True, check=True, env=dict(os.environ)
        )
    except subprocess.CalledProcessError as e:
        print(f"❌ Command failed with exit code {e.returncode}")
        raise
    

def write_json(file_path, data, jsonl=False):
    with open(file_path, "wb") as file:
        if jsonl:
            file.write(encoder.encode_lines(data))
        else:
            file.write(format(encoder.encode(data)))
    print(f"The file contains {len(data)} items.")
    print("Saved to", file_path)


def read_json(file_path, jsonl=False):
    file_path = Path(file_path)
    if not file_path.is_file():
        raise ValueError("filepath is not a file")
    # if not file_path.suffix == ".jsonl" and jsonl:
    #     raise ValueError("the file is not jsonl")
    # if not file_path.suffix == ".json" and not jsonl:
    #     raise ValueError("the file is not json")
    file_path = file_path.__str__()

    output = []

    with open(file_path, "rb") as file:
        data = file.read()
        if jsonl:
            output = decoder.decode_lines(data)
        else:
            output = decoder.decode(data)

    print(f"The file is of type: {type(output)}")
    print(f"The file contains {len(output)} items.")
    return output


In [36]:
tnp_query = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/tmrl/temporal/temporal_nobel_prize/train/dev.jsonl", jsonl=True)

The file is of type: <class 'list'>
The file contains 306 items.


In [38]:
import re

# Define regexes to classify the STRUCTURE of already-extracted temporal strings.
# These assume the input 'text' is just the temporal phrase (e.g., "from 1990 to 2000")
# not the whole query.
temporal_regexes = {
    # "from [anything] to [anything]"
    "range_from_to": re.compile(r"^from\s+.+?\s+to\s+.+$", re.IGNORECASE),
    
    # "between [anything] and [anything]"
    "range_between_and": re.compile(r"^between\s+.+?\s+and\s+.+$", re.IGNORECASE),
    
    # "after [anything]"
    "after": re.compile(r"^after\s+.+$", re.IGNORECASE),
    
    # "before [anything]"
    "before": re.compile(r"^before\s+.+$", re.IGNORECASE),
    
    # "in early [anything]"
    "early": re.compile(r"^in\s+early\s+.+$", re.IGNORECASE),
    
    # "in late [anything]"
    "late": re.compile(r"^in\s+late\s+.+$", re.IGNORECASE),
    
    # "in [anything]" 
    # Note: This is checked last or specifically to catch simple "in 1990" 
    # that didn't match early/late.
    "single_year_in": re.compile(r"^in\s+.+$", re.IGNORECASE),
}

def check_temporal_structure(text, regex) -> bool:
    """
    Checks if the pre-extracted temporal text matches the specific structure regex.
    Returns True if it matches, False otherwise.
    """
    if not text:
        return False
    # Use fullmatch or anchors ^ $ in regex to ensure the whole string fits the pattern
    return bool(regex.match(str(text).strip()))

output_jsonl = {
    k: [] for k in temporal_regexes.keys()
}

for sample in tnp_query:
    query_id = sample['query_id']
    temporal = sample.get('temporal', "")
    query = sample['query']
    pos = sample["positive_passages"]
    neg = sample["negative_passages"]
    
    if temporal == "":
        continue
    
    # Iterate over each temporal pattern type
    for pattern_name, regex in temporal_regexes.items():
        new_pos = []
        new_neg = []
        
        # Filter Positive Passages
        for p in pos:
            # Safely get the first temporal string from the passage
            p_temp_list = p.get('temporal', [])
            pt = p_temp_list[0] if len(p_temp_list) > 0 else ""
            
            if pt:
                # Check if this passage's temporal info matches the current structure
                if check_temporal_structure(pt, regex):
                    new_pos.append(p)
        
        # Filter Negative Passages
        for p in neg:
            # Safely get the first temporal string from the passage
            p_temp_list = p.get('temporal', [])
            pt = p_temp_list[0] if len(p_temp_list) > 0 else ""
            
            if pt:
                # Check if this passage's temporal info matches the current structure
                if check_temporal_structure(pt, regex):
                    new_neg.append(p)
        
        # Only add to output if we have at least one positive AND one negative
        # that match this specific temporal structure
        if len(new_pos) > 0 and len(new_neg) > 0:
            output_jsonl[pattern_name].append(
                {
                    "query_id": query_id,
                    "query": query,
                    "temporal": temporal,
                    "positive_passages": new_pos,
                    "negative_passages": new_neg
                }
            )

# Optional: Print summary
for k, v in output_jsonl.items():
    print(f"Pattern '{k}': {len(v)} samples")

Pattern 'range_from_to': 227 samples
Pattern 'range_between_and': 135 samples
Pattern 'after': 111 samples
Pattern 'before': 23 samples
Pattern 'early': 0 samples
Pattern 'late': 0 samples
Pattern 'single_year_in': 128 samples


In [39]:
total = 0
for k, v in output_jsonl.items():
    total += len(v)
    print(k, len(v))

print(total)

range_from_to 227
range_between_and 135
after 111
before 23
early 0
late 0
single_year_in 128
624


In [40]:
for name, output in output_jsonl.items():
    write_json(f"/home/thuy0050/mg61_scratch2/thuy0050/data/tmrl/temporal/temporal_nobel_prize/train/tsm_traindev/dev_{name}_{len(output)}.jsonl", output, jsonl=True)

The file contains 227 items.
Saved to /home/thuy0050/mg61_scratch2/thuy0050/data/tmrl/temporal/temporal_nobel_prize/train/tsm_traindev/dev_range_from_to_227.jsonl
The file contains 135 items.
Saved to /home/thuy0050/mg61_scratch2/thuy0050/data/tmrl/temporal/temporal_nobel_prize/train/tsm_traindev/dev_range_between_and_135.jsonl
The file contains 111 items.
Saved to /home/thuy0050/mg61_scratch2/thuy0050/data/tmrl/temporal/temporal_nobel_prize/train/tsm_traindev/dev_after_111.jsonl
The file contains 23 items.
Saved to /home/thuy0050/mg61_scratch2/thuy0050/data/tmrl/temporal/temporal_nobel_prize/train/tsm_traindev/dev_before_23.jsonl
The file contains 0 items.
Saved to /home/thuy0050/mg61_scratch2/thuy0050/data/tmrl/temporal/temporal_nobel_prize/train/tsm_traindev/dev_early_0.jsonl
The file contains 0 items.
Saved to /home/thuy0050/mg61_scratch2/thuy0050/data/tmrl/temporal/temporal_nobel_prize/train/tsm_traindev/dev_late_0.jsonl
The file contains 128 items.
Saved to /home/thuy0050/mg61_sc